## Troubleshooting — application failures

~30% of the CKA is troubleshooting: a broken cluster, you fix it. The fast path is *knowing the failure modes* and the first command for each.

### Pod is `Pending`

```bash
kubectl describe pod <name>     # the Events section holds the answer
```

- **`FailedScheduling`** — no node fits: insufficient CPU/memory, untolerated taint, node-selector mismatch, volume-zone affinity.
- **`FailedMount`** — PVC not binding, or volume on another node.
- **`ImagePullBackOff` / `ErrImagePull`** — image missing or pull secret wrong.

### `CrashLoopBackOff`

```bash
kubectl logs <name> --previous   # the CRASHED container's logs — key
kubectl describe pod <name>      # restartCount + last exit code
```

- **App bug** — in the logs. **OOMKilled** — exit 137, over memory limit (`describe` shows `Reason: OOMKilled`). **Missing config** — broken ConfigMap/Secret ref. **Failing readiness** — not restarted, but `READY: 0/1` and no Service traffic.

### Pod can't reach a Service

```bash
kubectl get endpoints <name>     # <none> = selector mismatch or not Ready
kubectl describe svc <name>      # selector + addresses
# from a sibling: nslookup <svc>; nc -vz <svc> <port>
```

### `kubectl debug` — ephemeral containers

For minimal images (distroless/scratch): `kubectl debug -it <pod> --image=busybox:1.36 --target=<container>` shares the pod's process namespace. Node-level: `kubectl debug node/<name> -it --image=busybox` mounts the node root at `/host`. On our map this section frames the **Pods** box — where most "it's broken" tickets land, and the `describe`/`logs`/`debug` verbs that diagnose them.